# XGBoost price model: does adding LLM text features help?

Two feature sets are compared on identical splits:
- **Baseline**: structured listing features from `data/final_dataset.csv`
- **+ Gemini**: baseline plus 5 scores Gemini extracted from the listing description (`model/description.ipynb`)

Each set gets its own hyperparameter search on the training split. They are then compared with repeated K-fold CV (paired, same folds) and once on a held-out test set. The target is `log1p(lastSoldPrice)`; sales under $100k (non-market transfers) are excluded.

In [1]:
import numpy as np
import pandas as pd
import xgboost as xgb
from scipy.stats import randint, uniform, ttest_rel
from sklearn.model_selection import train_test_split, RandomizedSearchCV, RepeatedKFold, cross_val_score
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error, mean_squared_error, r2_score

SEED = 42

In [2]:
# Load the structured dataset and the Gemini features, joined on listing ID (never on row order)
df = pd.read_csv('../data/final_dataset.csv')
gemini_feats = pd.read_csv('../data/gemini_encoded_descriptions.csv')

gemini_cols = [f'gemini_{c}' for c in gemini_feats.columns if c != 'zpid']
gemini_feats.columns = ['zpid'] + gemini_cols

df = df.merge(gemini_feats, on='zpid', how='left', validate='one_to_one')
assert df[gemini_cols].notna().any(axis=1).sum() == gemini_feats[gemini_cols].notna().any(axis=1).sum(), 'Gemini rows failed to join'

print(f"Rows: {len(df):,}")
print(f"Listings without Gemini scores (empty description): {df[gemini_cols[0]].isna().sum()}")

Rows: 4,853
Listings without Gemini scores (empty description): 25


In [3]:
target_col = 'financial_lastSoldPrice'

# Sales under $100k in San Francisco are transfers (family, partial interest), not market sales
MIN_PRICE = 100_000
n_before = len(df)
df = df[df[target_col] >= MIN_PRICE].reset_index(drop=True)
print(f"Dropped {n_before - len(df)} non-market sales under ${MIN_PRICE:,}")

non_features = ['zpid', target_col, 'basic_info_description']

baseline_cols = [c for c in df.columns if c not in non_features + gemini_cols]
feature_sets = {
    'XGBoost (baseline)': baseline_cols,
    'XGBoost + Gemini': baseline_cols + gemini_cols,
}

y = np.log1p(df[target_col])

# 80/20 split; the test set is only used once, at the very end
train_idx, test_idx = train_test_split(df.index, test_size=0.2, random_state=SEED, shuffle=True)
y_train, y_test = y.loc[train_idx], y.loc[test_idx]

print(f"Baseline features: {len(baseline_cols)}, Gemini features: {len(gemini_cols)}")
print(f"Train: {len(train_idx):,}  Test: {len(test_idx):,}")

Dropped 15 non-market sales under $100,000
Baseline features: 107, Gemini features: 5
Train: 3,870  Test: 968


In [4]:
# Hyperparameter search, run separately for each feature set on the training split only
param_distributions = {
    'max_depth': randint(2, 6),
    'learning_rate': uniform(0.01, 0.09),
    'n_estimators': randint(300, 1500),
    'min_child_weight': randint(1, 30),
    'subsample': uniform(0.5, 0.5),
    'colsample_bytree': uniform(0.3, 0.7),
    'reg_alpha': uniform(0, 5),
    'reg_lambda': uniform(1, 20),
    'gamma': uniform(0, 1),
}

best_params = {}
for name, cols in feature_sets.items():
    search = RandomizedSearchCV(
        xgb.XGBRegressor(objective='reg:squarederror', random_state=SEED, n_jobs=1),
        param_distributions=param_distributions,
        n_iter=40,
        cv=5,
        scoring='neg_root_mean_squared_error',
        random_state=SEED,
        n_jobs=-1,
    )
    search.fit(df.loc[train_idx, cols], y_train)
    best_params[name] = search.best_params_
    print(f"{name}: best CV RMSE (log) = {-search.best_score_:.4f}")

XGBoost (baseline): best CV RMSE (log) = 0.2527


XGBoost + Gemini: best CV RMSE (log) = 0.2439


In [5]:
# Paired comparison: both feature sets scored on the same 5x3 repeated K-fold splits of the training data
cv = RepeatedKFold(n_splits=5, n_repeats=3, random_state=SEED)
cv_rmse = {}
for name, cols in feature_sets.items():
    model = xgb.XGBRegressor(objective='reg:squarederror', random_state=SEED, n_jobs=-1, **best_params[name])
    cv_rmse[name] = -cross_val_score(model, df.loc[train_idx, cols], y_train, cv=cv,
                                     scoring='neg_root_mean_squared_error')
    print(f"{name}: CV RMSE (log) = {cv_rmse[name].mean():.4f} +/- {cv_rmse[name].std():.4f}")

base, gem = cv_rmse['XGBoost (baseline)'], cv_rmse['XGBoost + Gemini']
diff = gem - base
t_stat, p_value = ttest_rel(gem, base)
print(f"\nGemini - baseline: {diff.mean():+.4f} +/- {diff.std():.4f} (Gemini better in {(diff < 0).sum()}/{len(diff)} folds)")
print(f"Paired t-test p = {p_value:.3g} (folds overlap, so treat this as indicative, not exact)")

XGBoost (baseline): CV RMSE (log) = 0.2530 +/- 0.0175


XGBoost + Gemini: CV RMSE (log) = 0.2446 +/- 0.0174

Gemini - baseline: -0.0084 +/- 0.0031 (Gemini better in 14/15 folds)
Paired t-test p = 9.02e-08 (folds overlap, so treat this as indicative, not exact)


In [6]:
def evaluate_model(model, X, y_log):
    y_pred_log = model.predict(X)
    y_pred, y_actual = np.expm1(y_pred_log), np.expm1(y_log)
    return {
        'rmse_log': np.sqrt(mean_squared_error(y_log, y_pred_log)),
        'mae_usd': mean_absolute_error(y_actual, y_pred),
        'mape': mean_absolute_percentage_error(y_actual, y_pred),
        'median_ape': np.median(np.abs(y_pred - y_actual) / y_actual),
        'r2_usd': r2_score(y_actual, y_pred),
    }

# Final fit on the full training split, scored once on the held-out test set
test_results = {}
for name, cols in feature_sets.items():
    model = xgb.XGBRegressor(objective='reg:squarederror', random_state=SEED, n_jobs=-1, **best_params[name])
    model.fit(df.loc[train_idx, cols], y_train)
    test_results[name] = evaluate_model(model, df.loc[test_idx, cols], y_test)

results = pd.DataFrame({
    name: {
        'CV RMSE (log)': f"{cv_rmse[name].mean():.4f} ± {cv_rmse[name].std():.4f}",
        'Test RMSE (log)': f"{r['rmse_log']:.4f}",
        'Test MAE ($)': f"{r['mae_usd']:,.0f}",
        'Test MAPE': f"{r['mape']:.1%}",
        'Test median APE': f"{r['median_ape']:.1%}",
        'Test R² ($)': f"{r['r2_usd']:.3f}",
    }
    for name, r in test_results.items()
}).T
print(results.to_string())

                      CV RMSE (log) Test RMSE (log) Test MAE ($) Test MAPE Test median APE Test R² ($)
XGBoost (baseline)  0.2530 ± 0.0175          0.2641      371,036     18.7%           11.8%       0.625
XGBoost + Gemini    0.2446 ± 0.0174          0.2601      359,525     18.0%           10.9%       0.645


In [7]:
# Write the comparison that README.md reports
with open('../results.txt', 'w') as f:
    f.write("Generated by model/xgboost.ipynb\n\n")
    f.write(results.to_string())
    f.write(f"\n\nPaired CV difference (Gemini - baseline, log RMSE): {diff.mean():+.4f} +/- {diff.std():.4f}, "
            f"Gemini better in {(diff < 0).sum()}/{len(diff)} folds, paired t-test p = {p_value:.3g}\n")
print(open('../results.txt').read())

Generated by model/xgboost.ipynb

                      CV RMSE (log) Test RMSE (log) Test MAE ($) Test MAPE Test median APE Test R² ($)
XGBoost (baseline)  0.2530 ± 0.0175          0.2641      371,036     18.7%           11.8%       0.625
XGBoost + Gemini    0.2446 ± 0.0174          0.2601      359,525     18.0%           10.9%       0.645

Paired CV difference (Gemini - baseline, log RMSE): -0.0084 +/- 0.0031, Gemini better in 14/15 folds, paired t-test p = 9.02e-08

